# Notebook 18 — Hierarchical Drift Forecasting

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 17 built hierarchical prototype routing:

```text
parent route → child prototype → execution policy
```

Notebook 18 forecasts parent-route transitions before instability arrives.

Constraint view:
> hierarchical routing becomes more useful when route pressure can be forecast before drift alarms fire.

## Goals

1. Load Notebook 17 hierarchical routing outputs when available.
2. Estimate parent-route transition probabilities.
3. Compute route persistence and route pressure.
4. Forecast next parent route.
5. Score early-warning windows before parent transitions.
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 17 hierarchical route table

If the table is not available, create fallback hierarchical-route data.

In [ ]:
route_path = RESULTS_DIR / "notebook17_hierarchical_prototype_routing.csv"
parent_path = RESULTS_DIR / "notebook17_parent_route_summary.csv"

if route_path.exists():
    routes = pd.read_csv(route_path)
    print("Loaded:", route_path)
else:
    routes = None

if parent_path.exists():
    parent_summary = pd.read_csv(parent_path)
    print("Loaded:", parent_path)
else:
    parent_summary = None

if routes is None:
    print("Notebook 17 outputs not found; creating fallback route timeline.")
    rng = np.random.default_rng(42)
    n = 240
    parent_seq = []
    child_seq = []
    policy_seq = []
    blocks = [
        ("parent_0", "learned_drift_prototype", "prototype_recovery", 55),
        ("parent_1", "sequential_ids", "hybrid", 25),
        ("parent_2", "low_entropy_repeating", "coherent_local", 35),
        ("parent_0", "zipfian_smallints", "hybrid", 30),
        ("parent_1", "uniform_32bit", "simd", 45),
        ("parent_2", "clustered_ranges", "guarded_fallback", 50),
    ]
    while len(parent_seq) < n:
        for p, c, pol, L in blocks:
            for _ in range(L):
                if len(parent_seq) >= n:
                    break
                parent_seq.append(p if rng.random() > 0.08 else rng.choice(["parent_0", "parent_1", "parent_2"]))
                child_seq.append(c)
                policy_seq.append(pol)
            if len(parent_seq) >= n:
                break

    routes = pd.DataFrame({
        "window_id": np.arange(n),
        "parent_route": parent_seq,
        "child_route": child_seq,
        "updated_policy": policy_seq,
    })
    routes["parent_changed"] = routes["parent_route"].ne(routes["parent_route"].shift(1)).fillna(False)
    routes["child_changed"] = routes["child_route"].ne(routes["child_route"].shift(1)).fillna(False)
    routes["policy_changed"] = routes["updated_policy"].ne(routes["updated_policy"].shift(1)).fillna(False)
    routes["parent_switch_rate"] = routes["parent_changed"].rolling(15, min_periods=1).mean()
    routes["child_switch_rate"] = routes["child_changed"].rolling(15, min_periods=1).mean()
    routes["policy_switch_rate"] = routes["policy_changed"].rolling(15, min_periods=1).mean()
    routes["hierarchical_stability_score"] = (
        1 - 0.5*routes["parent_switch_rate"] - 0.3*routes["child_switch_rate"] - 0.2*routes["policy_switch_rate"]
    ).clip(0, 1)

work = routes.copy().sort_values("window_id").reset_index(drop=True)

for col in ["parent_route", "child_route", "updated_policy"]:
    if col not in work.columns:
        work[col] = "unknown"

if "parent_changed" not in work.columns:
    work["parent_changed"] = work["parent_route"].ne(work["parent_route"].shift(1)).fillna(False)
if "parent_switch_rate" not in work.columns:
    work["parent_switch_rate"] = work["parent_changed"].rolling(15, min_periods=1).mean()
if "child_switch_rate" not in work.columns:
    work["child_switch_rate"] = work["child_route"].ne(work["child_route"].shift(1)).rolling(15, min_periods=1).mean()
if "policy_switch_rate" not in work.columns:
    work["policy_switch_rate"] = work["updated_policy"].ne(work["updated_policy"].shift(1)).rolling(15, min_periods=1).mean()
if "hierarchical_stability_score" not in work.columns:
    work["hierarchical_stability_score"] = (
        1 - 0.5*work["parent_switch_rate"] - 0.3*work["child_switch_rate"] - 0.2*work["policy_switch_rate"]
    ).clip(0, 1)

work.head()

## Transition matrix for parent routes

Transition probability:

```text
P(next parent route | current parent route)
```

In [ ]:
parents = sorted(work["parent_route"].unique())
parent_to_idx = {p: i for i, p in enumerate(parents)}

counts = pd.DataFrame(0, index=parents, columns=parents, dtype=float)

for a, b in zip(work["parent_route"].iloc[:-1], work["parent_route"].iloc[1:]):
    counts.loc[a, b] += 1

transition_probs = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

counts, transition_probs

## Route persistence and pressure

- **Persistence:** probability of staying in the same parent route.
- **Route pressure:** probability of leaving the current parent route.

In [ ]:
persistence = {}
pressure = {}

for p in parents:
    stay = float(transition_probs.loc[p, p]) if p in transition_probs.index else 0.0
    persistence[p] = stay
    pressure[p] = 1.0 - stay

work["route_persistence"] = work["parent_route"].map(persistence).fillna(0.0)
work["route_pressure"] = work["parent_route"].map(pressure).fillna(1.0)

work[["window_id", "parent_route", "route_persistence", "route_pressure"]].head()

## Forecast next parent route

Forecast is the highest-probability transition from current parent route.

In [ ]:
def forecast_next_parent(parent):
    if parent not in transition_probs.index:
        return "unknown"
    row = transition_probs.loc[parent]
    if row.sum() <= 0:
        return parent
    return str(row.idxmax())

work["forecast_next_parent_route"] = work["parent_route"].apply(forecast_next_parent)
work["actual_next_parent_route"] = work["parent_route"].shift(-1).fillna(work["parent_route"])
work["forecast_correct"] = work["forecast_next_parent_route"] == work["actual_next_parent_route"]

forecast_accuracy = float(work["forecast_correct"].iloc[:-1].mean()) if len(work) > 1 else 0.0

forecast_accuracy, work[["window_id", "parent_route", "forecast_next_parent_route", "actual_next_parent_route", "forecast_correct"]].head()

## Early-warning score

A window is an early-warning candidate when:

- route pressure is high,
- hierarchy stability is dropping,
- child or policy switching rises before parent switch.

In [ ]:
def norm01(s, invert=False):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        vals = pd.Series(np.zeros(len(s)), index=s.index)
    else:
        vals = (s - lo) / (hi - lo)
    if invert:
        vals = 1.0 - vals
    return vals

work["stability_drop"] = (work["hierarchical_stability_score"].shift(1) - work["hierarchical_stability_score"]).fillna(0).clip(lower=0)
work["child_policy_pressure"] = (0.55 * work["child_switch_rate"] + 0.45 * work["policy_switch_rate"]).clip(0, 1)

work["early_warning_score"] = (
    0.35 * norm01(work["route_pressure"]) +
    0.30 * norm01(work["stability_drop"]) +
    0.25 * norm01(work["child_policy_pressure"]) +
    0.10 * norm01(work["parent_switch_rate"])
).clip(0, 1)

warning_threshold = float(work["early_warning_score"].quantile(0.85))
work["early_warning"] = work["early_warning_score"] >= warning_threshold

work[["window_id", "parent_route", "early_warning_score", "early_warning", "parent_changed"]].head()

## Forecast horizon check

For each parent transition, check whether an early warning occurred in the preceding horizon.

In [ ]:
horizon = 8

transition_windows = work.index[work["parent_changed"].astype(bool)].tolist()

transition_rows = []
for idx in transition_windows:
    if idx == 0:
        continue
    start = max(0, idx - horizon)
    prior = work.iloc[start:idx]
    warned = bool(prior["early_warning"].any())
    max_prior_score = float(prior["early_warning_score"].max()) if len(prior) else 0.0
    transition_rows.append({
        "transition_window": int(work.loc[idx, "window_id"]),
        "from_parent": str(work.loc[idx - 1, "parent_route"]),
        "to_parent": str(work.loc[idx, "parent_route"]),
        "warning_in_prior_horizon": warned,
        "max_prior_warning_score": max_prior_score,
        "horizon": horizon,
    })

transition_eval = pd.DataFrame(transition_rows)
early_warning_recall = float(transition_eval["warning_in_prior_horizon"].mean()) if len(transition_eval) else 0.0

transition_eval.head(), early_warning_recall

## Parent-route forecast confidence

Confidence is the max transition probability from current parent route.

In [ ]:
def forecast_confidence(parent):
    if parent not in transition_probs.index:
        return 0.0
    row = transition_probs.loc[parent]
    return float(row.max()) if row.sum() > 0 else 0.0

work["forecast_confidence"] = work["parent_route"].apply(forecast_confidence)
work["route_uncertainty"] = 1.0 - work["forecast_confidence"]

work[["window_id", "parent_route", "forecast_next_parent_route", "forecast_confidence", "route_uncertainty"]].head()

## Export forecasting tables

In [ ]:
csv_path = RESULTS_DIR / "notebook18_hierarchical_drift_forecasting.csv"
json_path = RESULTS_DIR / "notebook18_hierarchical_drift_forecasting.json"
transition_csv_path = RESULTS_DIR / "notebook18_parent_transition_matrix.csv"
transition_eval_csv_path = RESULTS_DIR / "notebook18_transition_warning_eval.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
transition_probs.to_csv(transition_csv_path)
transition_eval.to_csv(transition_eval_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", transition_csv_path)
print("Saved:", transition_eval_csv_path)

## Figure 1 — Parent transition probability matrix

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook18_parent_transition_matrix.png"

plt.figure(figsize=(7, 6))
plt.imshow(transition_probs.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(parents)), parents, rotation=45, ha="right")
plt.yticks(range(len(parents)), parents)
plt.colorbar(label="Transition probability")
plt.xlabel("Next parent route")
plt.ylabel("Current parent route")
plt.title("Hierarchical Drift Forecasting: Parent Transition Matrix")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Route pressure timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook18_route_pressure_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["route_pressure"], label="route pressure")
plt.plot(work["window_id"], work["route_uncertainty"], label="route uncertainty")
plt.xlabel("Window")
plt.ylabel("Score")
plt.title("Hierarchical Drift Forecasting: Route Pressure / Uncertainty")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Early-warning score timeline

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook18_early_warning_score_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["early_warning_score"], label="early-warning score")
plt.axhline(warning_threshold, linestyle="--", label="warning threshold")
warning_rows = work[work["early_warning"]]
plt.scatter(warning_rows["window_id"], warning_rows["early_warning_score"], s=30, label="warnings")
transition_rows_plot = work[work["parent_changed"]]
plt.scatter(transition_rows_plot["window_id"], transition_rows_plot["early_warning_score"], s=50, marker="x", label="parent transitions")
plt.xlabel("Window")
plt.ylabel("Score")
plt.title("Hierarchical Drift Forecasting: Early-Warning Score")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Forecast correctness over time

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook18_forecast_correctness_timeline.png"

rolling_acc = work["forecast_correct"].astype(float).rolling(20, min_periods=1).mean()

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], rolling_acc)
plt.xlabel("Window")
plt.ylabel("Rolling forecast accuracy")
plt.title("Hierarchical Drift Forecasting: Rolling Forecast Correctness")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Parent route actual vs forecast

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook18_parent_route_actual_vs_forecast.png"

labels = sorted(set(work["parent_route"]).union(set(work["forecast_next_parent_route"])))
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 5))
plt.step(work["window_id"], work["actual_next_parent_route"].map(lab_to_id), where="mid", label="actual next")
plt.step(work["window_id"], work["forecast_next_parent_route"].map(lab_to_id), where="mid", linestyle="--", label="forecast next")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Parent route")
plt.title("Hierarchical Drift Forecasting: Actual vs Forecast Parent Route")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Transition warning recall

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook18_transition_warning_recall.png"

if len(transition_eval):
    counts = transition_eval["warning_in_prior_horizon"].value_counts().rename_axis("warning").reset_index(name="count")
    counts["warning"] = counts["warning"].astype(str)
else:
    counts = pd.DataFrame({"warning": ["False"], "count": [0]})

plt.figure(figsize=(7, 4))
plt.bar(counts["warning"], counts["count"])
plt.xlabel("Warning before transition")
plt.ylabel("Transition count")
plt.title("Hierarchical Drift Forecasting: Warning Recall Before Transitions")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Figure 7 — Parent persistence by route

In [ ]:
fig_path_7 = FIGURES_DIR / "notebook18_parent_persistence_by_route.png"

persistence_df = pd.DataFrame({
    "parent_route": list(persistence.keys()),
    "persistence": list(persistence.values()),
    "pressure": [pressure[p] for p in persistence.keys()],
})

plt.figure(figsize=(8, 5))
plt.bar(persistence_df["parent_route"], persistence_df["persistence"])
plt.xlabel("Parent route")
plt.ylabel("Persistence probability")
plt.title("Hierarchical Drift Forecasting: Parent Persistence")
plt.tight_layout()
plt.savefig(fig_path_7, dpi=160)
plt.show()

print("Saved:", fig_path_7)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_18_hierarchical_drift_forecasting.md"

summary = {
    "windows": int(len(work)),
    "parent_route_count": int(len(parents)),
    "forecast_accuracy": forecast_accuracy,
    "early_warning_threshold": warning_threshold,
    "early_warning_windows": int(work["early_warning"].sum()),
    "parent_transition_count": int(work["parent_changed"].sum()),
    "early_warning_recall_prior_horizon": early_warning_recall,
    "forecast_horizon": int(horizon),
    "mean_route_pressure": float(work["route_pressure"].mean()),
    "mean_route_uncertainty": float(work["route_uncertainty"].mean()),
}

lines = [
    "# Report 18 — Hierarchical Drift Forecasting",
    "",
    "This report forecasts parent-route transitions and identifies early-warning windows before hierarchical drift.",
    "",
    "Constraint view:",
    "> hierarchical routing becomes more useful when route pressure can be forecast before drift alarms fire.",
    "",
    "## Generated outputs",
    "",
    f"- Forecasting CSV: `{csv_path}`",
    f"- Forecasting JSON: `{json_path}`",
    f"- Transition matrix CSV: `{transition_csv_path}`",
    f"- Transition warning evaluation CSV: `{transition_eval_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    f"- Figure: `{fig_path_7}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Parent transition probabilities",
    "",
    transition_probs.to_markdown(),
    "",
    "## Parent transition warning evaluation",
    "",
    transition_eval.to_markdown(index=False) if len(transition_eval) else "No parent transitions detected.",
    "",
    "## Interpretation",
    "",
    "- Parent-route persistence estimates which coarse routing states are stable.",
    "- Route pressure estimates likelihood that the current parent route will change.",
    "- Early-warning windows flag route instability before parent-route transitions.",
    "- Forecast confidence distinguishes stable routing from uncertain transition regions.",
    "",
    "## Next step",
    "",
    "Notebook 19 can perform recursive memory compression: merge redundant prototypes while preserving routing and reconstruction quality.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook18_hierarchical_drift_forecasting_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook18_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_18_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))